# 04 — MICM Fuel Prices | Silver Transformation

Turn the published weekly table (one column per fuel) into a tidy, validated series: **one row per price week × fuel**.

| Rule | Treatment |
|---|---|
| Week dates | Built from `DIA DESDE`/`DIA HASTA` + month label. The label is sometimes the start month and sometimes the end month of a week that crosses months, and has typos (`Febreo`, trailing spaces); resolved with the shared parser in `src/shared/micm_parsers.py`. |
| Zero prices | Treated as **not published** (`price_dop` NULL, `is_published = false`), e.g. Gasoil Óptimo before it was sold (2010–2012). Never used as a real price of zero. |
| Repeated weeks | Four weeks (9 Dec 2023 to 5 Jan 2024) appear twice in the file with some different values. The **later row in the file** is kept; earlier copies go to `quality.dq_rejected_records`. |
| Format reconciliation | Every CSV value must equal the XLSX value rounded to whole pesos (observational check). |
| Gaps and overlaps | Irregular weeks in the source are reported (observational), not altered. |

Target: `rd_economic_intelligence.silver.micm_fuel_prices_weekly` · full refresh · **Status:** Development.

In [0]:
import sys, uuid, json, importlib
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pyspark.sql import functions as F

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import micm_parsers as MP
importlib.reload(MP)

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.bronze.micm_fuel_prices_raw"
TARGET = f"{CATALOG}.silver.micm_fuel_prices_weekly"
REJECTED = f"{CATALOG}.quality.dq_rejected_records"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
KEY_COLS = ["DIA DESDE", "DIA HASTA", "MES", "AÑO"]
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

In [0]:
bronze = spark.table(SOURCE)
latest = bronze.groupBy("_source_file").agg(F.max("_ingestion_ts").alias("_ingestion_ts"))
bronze = bronze.join(latest, ["_source_file", "_ingestion_ts"])
def rows_of(fmt):
    return [(r["_source_row_number"], json.loads(r["record_json"]), r["_batch_id"], r["_source_file"])
            for r in bronze.filter(F.col("_source_format") == fmt).orderBy("_source_row_number").collect()]
xlsx, csv = rows_of("xlsx"), rows_of("csv")
fuel_columns = [c for c in xlsx[0][1].keys() if c not in KEY_COLS]
print(f"XLSX rows: {len(xlsx)} · CSV rows: {len(csv)} · fuel columns: {len(fuel_columns)}")

In [0]:
# Resolve week dates (file order matters)
periods = MP.resolve_periods([(r[1]["DIA DESDE"], r[1]["DIA HASTA"], r[1]["MES"], r[1]["AÑO"]) for r in xlsx])
flags = {}
for (_, _, f) in periods: flags[f] = flags.get(f, 0) + 1
print("Period resolution:", flags)

# Keep the last occurrence of each start date; earlier copies are rejected
last_row_for_start = {}
for idx, (s, e, f) in enumerate(periods):
    if s is not None: last_row_for_start[s] = idx

records, rejects = [], []
for idx, ((row_no, rec, batch, src), (s, e, f)) in enumerate(zip(xlsx, periods)):
    base = {"row": row_no, "period": f"{rec['DIA DESDE']}-{rec['DIA HASTA']} {rec['MES']} {rec['AÑO']}"}
    if s is None:
        rejects.append(("unresolvable_period", base, batch)); continue
    if last_row_for_start[s] != idx:
        rejects.append(("repeated_period_superseded", {**base, "kept_row": xlsx[last_row_for_start[s]][0]}, batch)); continue
    for col in fuel_columns:
        raw = rec[col]
        try:
            val = float(raw) if raw not in (None, "") else None
        except ValueError:
            rejects.append(("non_numeric_price", {**base, "fuel": col, "value": raw}, batch)); continue
        published = val is not None and val > 0
        records.append((s, e, f, MP.fuel_code(col), col.strip(), val if published else None, published, src, row_no, batch))

print(f"Silver records: {len(records):,} · rejected rows: {len(rejects)}")

In [0]:
silver = (spark.createDataFrame(records,
            "period_start_date date, period_end_date date, period_flag string, fuel_code string, fuel_name string, "
            "price_dop double, is_published boolean, _source_file string, _source_row_number int, _bronze_batch_id string")
          .withColumn("price_dop", F.col("price_dop").cast("decimal(10,2)"))
          .withColumn("price_unit", F.lit("DOP per gallon"))
          .withColumn("year", F.year("period_start_date"))
          .withColumn("_silver_run_id", F.lit(RUN_ID))
          .withColumn("_silver_processed_ts", F.lit(RUN_TS).cast("timestamp")))

# Data-quality rules
checks = []
def add(name, rtype, tested, failed):
    checks.append((RUN_ID, TARGET, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))
n = silver.count()
add("unique_week_fuel", "enforced", n, n - silver.select("period_start_date", "fuel_code").distinct().count())
add("end_after_start", "enforced", n, silver.filter(F.col("period_end_date") < F.col("period_start_date")).count())
add("published_price_positive", "enforced", n, silver.filter(F.col("is_published") & ~(F.col("price_dop") > 0)).count())
add("no_unresolvable_periods", "enforced", len(xlsx), sum(1 for r in rejects if r[0] == "unresolvable_period"))
add("repeated_periods_superseded", "observational", len(xlsx), sum(1 for r in rejects if r[0] == "repeated_period_superseded"))
add("weekly_continuity_gaps_overlaps", "observational", len(periods), flags.get("gap", 0) + flags.get("overlap", 0))
add("zero_prices_as_not_published", "observational", n, silver.filter(~F.col("is_published")).count())
# CSV = XLSX rounded to whole pesos
mism = 0; tested = 0
for (rx, rec_x, _, _), (rc, rec_c, _, _) in zip(xlsx, csv):
    for col in fuel_columns:
        cv = rec_c.get(col) if col in rec_c else rec_c.get(col + " ")
        if cv in (None, "") or rec_x[col] in (None, ""): continue
        tested += 1
        if Decimal(str(rec_x[col])).quantize(Decimal("1"), ROUND_HALF_UP) != Decimal(str(cv)).quantize(Decimal("1")):
            if abs(float(rec_x[col]) - float(cv)) > 0.5: mism += 1
add("csv_equals_xlsx_rounded", "observational", tested, mism)

for c in checks: print(f"  {c[6]:<4} {c[2]:<34} tested={c[4]:,} failed={c[5]:,}")
assert not [c for c in checks if c[6] == "FAIL"], "Enforced rules failed; nothing written"

In [0]:
silver.orderBy("fuel_code", "period_start_date").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Silver: MICM weekly fuel prices, one row per price week and fuel, DOP per gallon. Zero = not published (NULL). Repeated weeks resolved to the later file row.'")
spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN period_start_date SET NOT NULL")
spark.sql(f"ALTER TABLE {TARGET} DROP CONSTRAINT IF EXISTS chk_price_positive")
spark.sql(f"ALTER TABLE {TARGET} ADD CONSTRAINT chk_price_positive CHECK (price_dop IS NULL OR price_dop > 0)")
if rejects:
    spark.createDataFrame([(RUN_ID, TARGET, r[0], r[1]["period"], json.dumps(r[1], ensure_ascii=False, default=str), r[2], RUN_TS) for r in rejects],
        "run_id string, target_table string, rule_name string, record_key string, record_json string, source_batch_id string, detected_ts timestamp").write.mode("append").saveAsTable(REJECTED)
spark.createDataFrame(checks, "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

s = spark.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT fuel_code) fuels, MIN(period_start_date) a, MAX(period_end_date) b FROM {TARGET}").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "rows": s["n"], "fuels": s["fuels"], "first": str(s["a"]), "last": str(s["b"]),
    "period_flags": flags, "rejected": len(rejects), "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks}}))